# Broomstick Model - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC simulation results for a broomstick (cylinder) geometry using the ENDF/B-VIII.1 nuclear data library for photon transport.

## Model Description

- **Geometry**: Cylinder with radius 1 cm, height 100 cm (z = -50 to +50), vacuum boundaries
- **Source**: Point source at origin, 1 MeV monoenergetic photons, isotropic
- **Material**: Single element at 1.0 g/cm³, temperature 294 K
- **Statistics**: 5,000,000 particles × 10 batches = 50,000,000 total histories
- **Tallies**: Photon flux (energy-binned, CCFE-24-PHOTON), coherent-scatter, incoherent-scatter, photoelectric, pair-production

In [ ]:
# Setup and imports
import numpy as np
import pandas as pd
import glob
import os
import yamc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.1": "/home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/",
}

NUCLEAR_DATA = "endf-b8.1"
PARTICLES = 5000000
BATCHES = 10
SOURCE_ENERGY = 1e6  # 1 MeV photons
MAX_PLOT_ENERGY = SOURCE_ENERGY * 1.1
FLUX_DECADES = 6

# Optional smoke-test cap on the iteration list (env var; unset/0 = full).
NUMBER_OF_SIMULATIONS = int(os.environ.get("YAMC_VERIFY_NUMBER_OF_SIMULATIONS", "0")) or None

# Codes/backends to run. Comma-separated list via the MC_BACKENDS env var.
# Tokens:
#   openmc-cpu  — reference OpenMC simulation
#   yamc-cpu    — YAMC on CPU
#   yamc-gpu    — YAMC on GPU (requires a working f64 Vulkan adapter)
# Tokens are independent — any one, two, or all three are valid.
# Default runs all three. Examples:
#   MC_BACKENDS=yamc-cpu,yamc-gpu      # skip OpenMC (compare yamc backends)
#   MC_BACKENDS=openmc-cpu,yamc-cpu    # no GPU
_VALID_BACKENDS = ("openmc-cpu", "yamc-cpu", "yamc-gpu")
_raw = os.environ.get("MC_BACKENDS", "openmc-cpu,yamc-cpu,yamc-gpu")
MC_BACKENDS = [b.strip().lower() for b in _raw.split(",") if b.strip()]
_bad = [b for b in MC_BACKENDS if b not in _VALID_BACKENDS]
if _bad:
    raise ValueError(
        f"MC_BACKENDS contains unknown token(s) {_bad}; "
        f"valid tokens are {list(_VALID_BACKENDS)}"
    )
if not MC_BACKENDS:
    raise ValueError("MC_BACKENDS resolved to an empty list")
RUN_OPENMC = "openmc-cpu" in MC_BACKENDS
COMPUTES = [b.split("-", 1)[1] for b in MC_BACKENDS if b.startswith("yamc-")]
# No precheck for GPU availability — the dispatch raises a clean
# RuntimeError if no f64 Vulkan adapter is present, which is enough
# signal for both CI and interactive use.
print(f"Backends: {MC_BACKENDS}")

# Cylinder dimensions
CYLINDER_RADIUS = 1.0  # cm
CYLINDER_HALF_HEIGHT = 50.0  # cm (total height = 100 cm)

# CCFE-24-PHOTON energy bins (eV)
PHOTON_BINS = [
    1.0e3, 5.0e3, 1.0e4, 2.0e4, 5.0e4, 1.0e5, 2.0e5, 3.0e5, 4.0e5,
    5.0e5, 5.11e5, 6.0e5, 8.0e5, 1.0e6, 1.5e6, 2.0e6, 3.0e6, 4.0e6,
    5.0e6, 6.0e6, 8.0e6, 1.0e7, 1.2e7, 1.5e7, 2.0e7,
]

In [ ]:
# OpenMC results cache — load reference results from disk if a previous
# run produced them with the same parameters; otherwise run OpenMC and
# write the results out for next time.
import sys
from pathlib import Path as _Path

_repo_root = _Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "cache_utils.py").exists():
    _repo_root = _repo_root.parent
if (_repo_root / "cache_utils.py").exists() and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from cache_utils import cached_or_run

CACHE_DIR = _Path("cache")
OPENMC_CACHE_META = {
    "library": NUCLEAR_DATA,
    "particles": PARTICLES,
    "batches": BATCHES,
    "source_energy": SOURCE_ENERGY,
    "cylinder_radius": CYLINDER_RADIUS,
    "cylinder_half_height": CYLINDER_HALF_HEIGHT,
}

In [ ]:
def run_openmc_simulation(element, nuclear_data):
    """Run OpenMC photon simulation for a single element."""
    import openmc as mc

    mc.reset_auto_ids()
    mc.config['cross_sections'] = f"{NUCLEAR_DATA_DIR[nuclear_data]}cross_sections.xml"

    # Geometry: broomstick cylinder
    cylinder = mc.ZCylinder(r=CYLINDER_RADIUS, boundary_type='vacuum')
    z_bot = mc.ZPlane(z0=-CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    z_top = mc.ZPlane(z0=CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    region = -cylinder & +z_bot & -z_top

    # Material
    material = mc.Material()
    material.add_element(element, 1.0)
    material.set_density("g/cm3", 1.0)
    material.temperature = 294

    cell = mc.Cell(name="broomstick", region=region, fill=material)
    geometry = mc.Geometry([cell])

    # Source: 1 MeV isotropic photons at origin
    source = mc.IndependentSource(
        space=mc.stats.Point([0, 0, 0]),
        energy=mc.stats.Discrete([SOURCE_ENERGY], [1.0]),
        angle=mc.stats.Isotropic(),
    )
    source.particle = "photon"

    settings = mc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1,
        run_mode='fixed source',
        output={'tallies': False, 'summary': False},
    )

    # Filters and tallies
    cell_filter = mc.CellFilter(cell)
    photon_filter = mc.ParticleFilter("photon")
    energy_filter = mc.EnergyFilter(PHOTON_BINS)
    energy_bins_2d = energy_filter.bins
    energy_bins = np.concatenate([[energy_bins_2d[0, 0]], energy_bins_2d[:, 1]])

    tallies = []

    # Photon interaction score tallies
    photon_scores = ['coherent-scatter', 'incoherent-scatter', 'photoelectric', 'pair-production']
    for score in ['flux'] + photon_scores:
        tally = mc.Tally()
        tally.filters = [cell_filter, photon_filter]
        tally.scores = [score]
        tally.name = score
        tallies.append(tally)

    # Energy-binned photon flux tally
    tally_spectrum = mc.Tally()
    tally_spectrum.filters = [cell_filter, photon_filter, energy_filter]
    tally_spectrum.scores = ['flux']
    tally_spectrum.name = "flux_energy_binned"
    tallies.append(tally_spectrum)

    # Run
    model = mc.Model(geometry=geometry, settings=settings, tallies=tallies)
    sp_filename = model.run(apply_tally_results=True)
    sp = mc.StatePoint(sp_filename)

    transport_time = sp.runtime['transport']
    total_particles = PARTICLES * BATCHES
    particles_per_second = total_particles / transport_time

    results = {
        'element': element,
        'code': 'openmc',
        'particles/s': particles_per_second,
        'tallies': {},
        'tally_std': {},
    }

    for tally in tallies:
        if tally.name != "flux_energy_binned":
            results['tallies'][tally.name] = tally.mean.flatten()[0]
            results['tally_std'][tally.name] = tally.std_dev.flatten()[0]

    results['flux_spectra_mean'] = np.array(tally_spectrum.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(tally_spectrum.std_dev).squeeze()
    results['energy_bins'] = energy_bins

    return results

In [ ]:
def run_yamc_simulation(element, nuclear_data, compute="cpu"):
    """Run YAMC photon simulation for a single element. Returns (results_dict, model).

    `compute` selects the transport backend ("cpu" or "gpu"). The GPU
    path routes via the new `multi_cell_photon_transport` kernel
    (Compton via Klein-Nishina, Rayleigh via form-factor sampling,
    photoelectric/pair as absorption — see TODO.md for what's
    intentionally not modelled on the GPU path yet).
    """
    import yamc as mc

    photon_path = f"{NUCLEAR_DATA_DIR[nuclear_data]}photon/{element}.h5"

    # Geometry: broomstick cylinder
    cylinder = mc.ZCylinder(radius=CYLINDER_RADIUS, boundary='vacuum')
    z_bot = mc.ZPlane(z0=-CYLINDER_HALF_HEIGHT, boundary='vacuum')
    z_top = mc.ZPlane(z0=CYLINDER_HALF_HEIGHT, boundary='vacuum')
    region = -cylinder & +z_bot & -z_top

    # Material - photon-only, no neutron data needed
    material = mc.Material(
        composition={element: 1.0},
        density=1.0,
        temperature=294,
        id=1,
    )
    material.read_nuclear_data('endf-b8.1')

    cell = mc.Cell(id=1, name="broomstick", region=region, material=material)
    geometry = mc.Geometry([cell])

    # Source: 1 MeV isotropic photons at origin
    source = mc.PhotonSource(
        position=mc.Point(0, 0, 0),
        direction=mc.Isotropic(),
        energy=mc.Discrete([SOURCE_ENERGY], [1.0]),
    )

    tallies = []

    # Photon interaction score tallies
    photon_scores = ['coherent-scatter', 'incoherent-scatter', 'photoelectric', 'pair-production']
    for score in ['flux'] + photon_scores:
        t = mc.Tally(
            scores=[str(score)],
            name=str(score),
            cells=cell,
            particle="photon",
        )
        tallies.append(t)

    # Energy-binned photon flux tally
    tally_spectrum = mc.Tally(
        scores=['flux'],
        name="flux_energy_binned",
        cells=cell,
        particle="photon",
        energy_group_structure='CCFE-24-PHOTON',
    )
    tallies.append(tally_spectrum)

    # Energy group bins (for plotting)
    energy_bins = np.array(tally_spectrum.energy_bins)

    # Run
    model = mc.Model(
        geometry=geometry,
        tallies=tallies,
        source=source,
        particles_per_batch=PARTICLES,
        batches=BATCHES,
        seed=1,
        photon_transport=True,
    )
    import time as _t
    _t0 = _t.perf_counter()
    sim_results = model.simulate_transport(compute=compute)
    _elapsed = _t.perf_counter() - _t0

    results = {
        'element': element,
        'code': f'yamc-{compute}',
        'particles/s': (
            model.particles_per_second
            if model.particles_per_second is not None
            else (PARTICLES * BATCHES) / _elapsed if _elapsed > 0 else 0
        ),
        'tallies': {},
        'tally_std': {},
    }

    for t in tallies:
        if t.name != "flux_energy_binned":
            r = sim_results[t]
            results['tallies'][t.name] = r.mean[0]
            results['tally_std'][t.name] = r.standard_deviation[0]

    spec = sim_results[tally_spectrum]
    results['flux_spectra_mean'] = np.array(spec.mean).squeeze()
    results['flux_spectra_std_dev'] = np.array(spec.standard_deviation).squeeze()
    results['energy_bins'] = energy_bins

    return results, model

In [ ]:
def create_step_trace(energy_bins, values, name, color, dash=None, showlegend=True, legend='legend'):
    """Create a step plot trace for Plotly (similar to matplotlib step with where='post')."""
    x_step = []
    y_step = []
    for i in range(len(values)):
        x_step.extend([energy_bins[i], energy_bins[i+1]])
        y_step.extend([values[i], values[i]])

    return go.Scatter(
        x=x_step,
        y=y_step,
        mode='lines',
        name=name,
        line=dict(color=color, width=2, dash=dash),
        showlegend=showlegend,
        legend=legend,
    )


# Per-backend plot styling. Keys are backend labels used in the legend.
BACKEND_STYLES = {
    'OpenMC': dict(color='blue', dash=None, marker='circle'),
    'YAMC (cpu)': dict(color='red', dash='dash', marker='square'),
    'YAMC (gpu)': dict(color='orange', dash='dot', marker='diamond'),
}
RATIO_COLORS = {
    'YAMC (cpu) / OpenMC': 'green',
    'YAMC (gpu) / OpenMC': 'purple',
    'YAMC (gpu) / YAMC (cpu)': 'teal',
}
RATIO_FILLS = {
    'green': 'rgba(0, 128, 0, 0.2)',
    'purple': 'rgba(128, 0, 128, 0.2)',
    'teal': 'rgba(0, 128, 128, 0.2)',
}


def create_comparison_plot(openmc_results, yamc_results_by_backend):
    """Create a Plotly subplot with photon flux spectra (top) and per-backend ratios (bottom).

    Parameters
    ----------
    openmc_results : dict or None
        OpenMC results dictionary. When None, the OpenMC trace is omitted
        and the ratio panel uses the first YAMC backend as reference. If
        only one YAMC backend is provided and openmc_results is None, the
        ratio panel is hidden.
    yamc_results_by_backend : dict[str, dict]
        Maps backend name (e.g. "cpu", "gpu") to a YAMC results dictionary.
    """

    if openmc_results is not None:
        ref_label = 'OpenMC'
        ratio_targets = [(f'YAMC ({b})', res) for b, res in yamc_results_by_backend.items()]
    elif len(yamc_results_by_backend) >= 2:
        backends = list(yamc_results_by_backend)
        ref_label = f'YAMC ({backends[0]})'
        ratio_targets = [(f'YAMC ({b})', yamc_results_by_backend[b]) for b in backends[1:]]
    else:
        ref_label = None
        ratio_targets = []

    spectra = []
    if openmc_results is not None:
        spectra.append(('OpenMC', openmc_results))
    for backend, res in yamc_results_by_backend.items():
        spectra.append((f'YAMC ({backend})', res))

    if not spectra:
        raise ValueError("create_comparison_plot needs at least one result")

    energy_bins = spectra[0][1]['energy_bins']
    bin_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])
    lethargy_bin_width = np.log10(energy_bins[1:] / energy_bins[:-1])

    flux_by_label = {}
    std_by_label = {}
    for label, res in spectra:
        flux_by_label[label] = res['flux_spectra_mean'].copy() / lethargy_bin_width
        std_by_label[label] = res['flux_spectra_std_dev'].copy() / lethargy_bin_width

    has_flux = np.zeros(len(energy_bins) - 1, dtype=bool)
    for f in flux_by_label.values():
        has_flux = has_flux | (f > 0)
    if np.any(has_flux):
        first_nonzero_idx = int(np.argmax(has_flux))
        min_plot_energy = energy_bins[first_nonzero_idx]
    else:
        min_plot_energy = energy_bins[0]

    all_positive_max = []
    for f in flux_by_label.values():
        if np.any(f > 0):
            all_positive_max.append(np.max(f[f > 0]))
    max_flux = max(all_positive_max) if all_positive_max else 1e-10
    flux_threshold = max_flux / (10 ** FLUX_DECADES)

    def _mask(values, std, ref):
        keep = ref >= flux_threshold
        return np.where(keep, values, np.nan), np.where(keep, std, np.nan)

    show_ratio = bool(ratio_targets)
    if show_ratio:
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.08,
            row_heights=[0.7, 0.3],
            subplot_titles=('Photon Flux per Unit Lethargy', f'Ratio (vs {ref_label})')
        )
    else:
        fig = make_subplots(
            rows=1, cols=1,
            subplot_titles=('Photon Flux per Unit Lethargy',),
        )

    for label, _res in spectra:
        f = flux_by_label[label]
        s = std_by_label[label]
        f_plot, s_plot = _mask(f, s, f)
        style = BACKEND_STYLES.get(label, dict(color='gray', dash=None, marker='circle'))
        fig.add_trace(
            create_step_trace(energy_bins, f_plot, label, style['color'], dash=style['dash']),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=bin_centers, y=f_plot, mode='markers',
                name=f'{label} error',
                marker=dict(color=style['color'], size=4, symbol=style['marker']),
                error_y=dict(type='data', array=s_plot, visible=True,
                             color=style['color'], thickness=1),
                showlegend=False,
            ),
            row=1, col=1
        )

    if show_ratio:
        ref_flux = flux_by_label[ref_label]
        ref_std = std_by_label[ref_label]
        for label, _res in ratio_targets:
            f = flux_by_label[label]
            s = std_by_label[label]
            valid = (ref_flux > 0) & (f > 0)
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio = np.where(valid, f / ref_flux, np.nan)
                rel_err_top = np.where(f > 0, s / f, 0)
                rel_err_ref = np.where(ref_flux > 0, ref_std / ref_flux, 0)
            ratio_rel_err = np.sqrt(rel_err_top**2 + rel_err_ref**2)
            ratio_std = np.where(valid, ratio * ratio_rel_err, np.nan)

            ratio_label = f'{label} / {ref_label}'
            ratio_color = RATIO_COLORS.get(ratio_label, 'green')
            rgba_fill = RATIO_FILLS.get(ratio_color, 'rgba(0, 128, 0, 0.2)')

            fig.add_trace(
                create_step_trace(energy_bins, ratio, ratio_label, ratio_color, legend='legend2'),
                row=2, col=1
            )
            fig.add_trace(
                go.Scatter(
                    x=np.concatenate([bin_centers, bin_centers[::-1]]),
                    y=np.concatenate([ratio + ratio_std, (ratio - ratio_std)[::-1]]),
                    fill='toself',
                    fillcolor=rgba_fill,
                    line=dict(color='rgba(255,255,255,0)'),
                    name=f'{ratio_label} uncertainty',
                    showlegend=False,
                ),
                row=2, col=1
            )
        fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)

    fig.update_xaxes(type="log", title_text="Energy (eV)",
                     range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
                     exponentformat='e', row=(2 if show_ratio else 1), col=1)
    if show_ratio:
        fig.update_xaxes(type="log",
                         range=[np.log10(min_plot_energy), np.log10(MAX_PLOT_ENERGY)],
                         exponentformat='e', row=1, col=1)
    fig.update_yaxes(type="log", title_text="Photon Flux / Lethargy", exponentformat='e', row=1, col=1)
    if show_ratio:
        fig.update_yaxes(title_text="Ratio", range=[0.5, 1.5], row=2, col=1)

    fig.update_layout(
        height=700 if show_ratio else 500,
        legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top'),
        legend2=dict(x=0.02, y=0.28, xanchor='left', yanchor='top'),
        margin=dict(l=60, r=20, t=40, b=40)
    )

    return fig


def calculate_spectral_agreement(ref_results, other_results, flux_threshold=1e-6):
    """Statistical agreement metrics comparing one photon spectrum to a reference."""

    ref_flux = ref_results['flux_spectra_mean']
    ref_std = ref_results['flux_spectra_std_dev']
    oth_flux = other_results['flux_spectra_mean']
    oth_std = other_results['flux_spectra_std_dev']

    chi_squared_terms = []
    rel_diffs = []
    weighted_rel_diffs = []
    flux_weights = []

    for i in range(len(ref_flux)):
        r = ref_flux[i]
        o = oth_flux[i]
        r_std = ref_std[i]
        o_std = oth_std[i]

        combined_std = np.sqrt(r_std**2 + o_std**2)
        if combined_std > 0:
            chi_squared_terms.append(((o - r) / combined_std)**2)

        if r > flux_threshold or o > flux_threshold:
            if r > 0:
                rel_diff = abs((o - r) / r) * 100
                if np.isfinite(rel_diff):
                    rel_diffs.append(rel_diff)
                    weight = max(r, o)
                    weighted_rel_diffs.append(rel_diff * weight)
                    flux_weights.append(weight)

    chi_squared = np.sum(chi_squared_terms)
    dof = len(chi_squared_terms)
    reduced_chi_squared = chi_squared / dof if dof > 0 else 0.0

    median_rel_diff = np.median(rel_diffs) if rel_diffs else 0.0
    weighted_avg_rel_diff = np.sum(weighted_rel_diffs) / np.sum(flux_weights) if flux_weights else 0.0

    if reduced_chi_squared < 1.5:
        quality = "Excellent"
    elif reduced_chi_squared < 3.0:
        quality = "Good"
    elif reduced_chi_squared < 5.0:
        quality = "Fair"
    else:
        quality = "Poor"

    return {
        'reduced_chi_squared': reduced_chi_squared,
        'degrees_of_freedom': dof,
        'median_rel_diff': median_rel_diff,
        'weighted_avg_rel_diff': weighted_avg_rel_diff,
        'quality': quality,
        'bins_analyzed': len(rel_diffs),
        'total_bins': len(ref_flux)
    }


def create_agreement_table(stats):
    """Create a summary table of spectral agreement statistics."""
    rows = [
        {'Metric': 'Reduced Chi-Squared', 'Value': f"{stats['reduced_chi_squared']:.2f}",
         'Note': f"(ideal ~ 1.0, dof={stats['degrees_of_freedom']})"},
        {'Metric': 'Weighted Avg Rel. Diff', 'Value': f"{stats['weighted_avg_rel_diff']:.2f}%",
         'Note': '(flux-weighted)'},
        {'Metric': 'Median Rel. Diff', 'Value': f"{stats['median_rel_diff']:.2f}%",
         'Note': '(robust to outliers)'},
        {'Metric': 'Agreement Quality', 'Value': stats['quality'],
         'Note': f"({stats['bins_analyzed']}/{stats['total_bins']} bins analyzed)"},
    ]
    return pd.DataFrame(rows)


# Photon interaction MT descriptions
PHOTON_MT_DESCRIPTIONS = {
    'flux': 'Total photon flux',
    'coherent-scatter': 'Coherent (Rayleigh) scatter (MT 502)',
    'incoherent-scatter': 'Incoherent (Compton) scatter (MT 504)',
    'photoelectric': 'Photoelectric absorption (MT 522)',
    'pair-production': 'Pair production (MT 516)',
}


def _format_tally(value, std):
    if value is None or np.isnan(value):
        return 'N/A'
    return f"{value:.4e} +/- {std:.2e}"


def _format_rel_diff(value, ref_val):
    if ref_val is None or np.isnan(ref_val) or ref_val == 0 or value is None or np.isnan(value):
        return 'N/A'
    return f"{(value - ref_val) / ref_val * 100:.2f}%"


def create_tally_table(openmc_results, yamc_results_by_backend):
    """Build a photon tally comparison table with one column per code/backend.

    Parameters
    ----------
    openmc_results : dict or None
        OpenMC results. When None, the OpenMC column is omitted and the
        first YAMC backend is used as the reference for relative differences.
    yamc_results_by_backend : dict[str, dict]
        Maps backend name (e.g. "cpu", "gpu") to a YAMC results dictionary.
    """

    all_tallies = set()
    if openmc_results is not None:
        all_tallies |= set(openmc_results['tallies'].keys())
    for res in yamc_results_by_backend.values():
        all_tallies |= set(res['tallies'].keys())

    yamc_backends = list(yamc_results_by_backend)
    if openmc_results is not None:
        ref_results = openmc_results
        rel_diff_targets = yamc_backends
    elif yamc_backends:
        ref_results = yamc_results_by_backend[yamc_backends[0]]
        rel_diff_targets = yamc_backends[1:]
    else:
        return pd.DataFrame()

    rows = []
    for name in sorted(all_tallies):
        if openmc_results is not None:
            openmc_val = openmc_results['tallies'].get(name, np.nan)
            openmc_std = openmc_results['tally_std'].get(name, np.nan)
        else:
            openmc_val = openmc_std = np.nan
        yamc_vals = {}
        for backend, res in yamc_results_by_backend.items():
            yamc_vals[backend] = (
                res['tallies'].get(name, np.nan),
                res['tally_std'].get(name, np.nan),
            )
        ref_val = ref_results['tallies'].get(name, np.nan)

        all_zero = (
            (openmc_results is None or openmc_val == 0 or np.isnan(openmc_val))
            and all((v[0] == 0 or np.isnan(v[0])) for v in yamc_vals.values())
        )
        if all_zero:
            continue

        reaction = PHOTON_MT_DESCRIPTIONS.get(name, name)
        row = {'Score': name, 'Reaction': reaction}
        if openmc_results is not None:
            row['OpenMC (cpu)'] = _format_tally(openmc_val, openmc_std)
        for backend, (val, std) in yamc_vals.items():
            row[f'YAMC ({backend})'] = _format_tally(val, std)
        for backend in rel_diff_targets:
            val, _ = yamc_vals[backend]
            row[f'Rel. Diff {backend} (%)'] = _format_rel_diff(val, ref_val)
        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# Discover elements with photon data available
photon_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'photon')

photon_files = sorted(glob.glob(os.path.join(photon_dir, '*.h5')))
elements = [os.path.splitext(os.path.basename(f))[0] for f in photon_files]

if not elements:
    raise FileNotFoundError(
        f"No photon data files found in {photon_dir}\n"
        f"Please check that the photon/ subdirectory exists."
    )

# Filter to elements that YAMC can construct from natural abundance.
# Synthetic/short-lived elements (Tc, Pm, Po, At, Rn, Fr, Ra, Ac, Np, Pu, Am, Cm, Bk, Cf, Es, Fm)
# have no natural abundance and cannot be expanded by composition={element: 1.0}.
supported = []
unsupported = []
for e in elements:
    try:
        yamc.Material(composition={e: 1.0}, density=1.0)
        supported.append(e)
    except Exception:
        unsupported.append(e)
elements = supported

if NUMBER_OF_SIMULATIONS:
    elements = elements[:NUMBER_OF_SIMULATIONS]

if unsupported:
    print(f"Skipping {len(unsupported)} elements without natural abundance: {unsupported}")
print(f"Running photon simulations for {len(elements)} elements: {elements}")

In [ ]:
# Run all simulations and collect results
all_results = {}
first_yamc_model = None

for element in elements:
    print(f"\nProcessing {element}...")
    per_element = {}
    yamc_ok = True
    for compute in COMPUTES:
        try:
            yamc_results, yamc_model = run_yamc_simulation(
                element, NUCLEAR_DATA, compute=compute,
            )
            per_element[f'yamc-{compute}'] = yamc_results
            if first_yamc_model is None:
                first_yamc_model = yamc_model
        except Exception as e:
            print(f"  YAMC ({compute}) Error: {e}")
            yamc_ok = False
            break
    if not yamc_ok:
        continue

    if RUN_OPENMC:
        try:
            openmc_results = cached_or_run(
                CACHE_DIR, NUCLEAR_DATA, "openmc", element,
                lambda el=element: run_openmc_simulation(el, NUCLEAR_DATA),
                meta=OPENMC_CACHE_META,
            )
            per_element['openmc'] = openmc_results
        except Exception as e:
            print(f"  OpenMC Error: {e}")
            continue

    all_results[element] = per_element
    if RUN_OPENMC:
        print(f"  OpenMC: {per_element['openmc']['particles/s']:.0f} particles/s")
    for compute in COMPUTES:
        yamc_pps = per_element[f'yamc-{compute}']['particles/s']
        print(f"  YAMC ({compute}): {yamc_pps:.0f} particles/s")

print(f"\nCompleted {len(all_results)} / {len(elements)} elements successfully.")

In [ ]:
# Display broomstick model geometry with source from first simulation
import base64
from IPython.display import HTML

if first_yamc_model is not None:
    html_plot = first_yamc_model.plot(basis="xz", n_samples=500, axis_units="cm")
    html_plot.save("broomstick_model_plot.html")
    encoded = base64.b64encode(html_plot.html.encode()).decode()
    display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="600" frameborder="0"></iframe>'))
else:
    print("No successful YAMC simulation to plot.")

## Results by Element

Below are the comparison results for each element. Each section includes:

- Interactive photon flux spectra plot (log-log scale)
- YAMC/OpenMC ratio plot with uncertainty band
- Photon tally comparison table (flux, coherent-scatter, incoherent-scatter, photoelectric, pair-production)
- Performance comparison (particles/second)

In [ ]:
# Display results for each element
for element, results in all_results.items():
    openmc_results = results.get('openmc')  # may be None when openmc-cpu not in MC_BACKENDS

    # Collect available YAMC backends in the order the user requested
    yamc_results_by_backend = {}
    for compute in COMPUTES:
        yres = results.get(f'yamc-{compute}')
        if yres is not None:
            yamc_results_by_backend[compute] = yres

    display(Markdown(f"## {element}"))

    # Performance line — one entry per code/backend that ran
    perf_parts = []
    if openmc_results is not None:
        perf_parts.append(f"OpenMC: {openmc_results['particles/s']:,.0f} particles/s")
    for compute, yres in yamc_results_by_backend.items():
        perf_parts.append(f"YAMC ({compute}): {yres['particles/s']:,.0f} particles/s")
    display(Markdown(f"**Performance:** " + " | ".join(perf_parts)))

    # Combined flux spectra plot
    fig = create_comparison_plot(openmc_results, yamc_results_by_backend)
    fig.update_layout(title=f"{element} - Photon Flux Spectra Comparison")
    fig.show()

    # Spectral agreement: each yamc backend vs the reference (openmc when
    # present, else the first yamc backend).
    if openmc_results is not None:
        ref_label = 'OpenMC'
        ref_results = openmc_results
        comparisons = list(yamc_results_by_backend.items())
    elif len(yamc_results_by_backend) >= 2:
        backends = list(yamc_results_by_backend)
        ref_label = f'YAMC ({backends[0]})'
        ref_results = yamc_results_by_backend[backends[0]]
        comparisons = [(b, yamc_results_by_backend[b]) for b in backends[1:]]
    else:
        comparisons = []

    for compute, yres in comparisons:
        heading = (
            f"#### Spectral Agreement (YAMC {compute} vs {ref_label})"
            if len(comparisons) > 1 or openmc_results is None
            else "#### Spectral Agreement"
        )
        display(Markdown(heading))
        stats = calculate_spectral_agreement(ref_results, yres)
        display(create_agreement_table(stats).style.hide(axis='index'))

    # Combined tally table
    display(Markdown("#### Photon Tally Comparison"))
    table = create_tally_table(openmc_results, yamc_results_by_backend)
    display(table.style.hide(axis='index'))

    print()  # spacing